# SASRec autoencoder Mel + Chroma

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, mixed_precision
import os
import gc

# ==========================================
# 1. ÚTVONALAK ÉS HIPERPARAMÉTEREK
# ==========================================
INPUT_PATH = "/content/drive/MyDrive/Diplomamunka"
AE_MATRIX_PATH = os.path.join(INPUT_PATH, "ae_vectors_closed_world_notempo.npy")  # 2 spektrogramos AE
TRAIN_DATA_PATH = os.path.join(INPUT_PATH, "train_pids_ae.npy")
VAL_DATA_PATH = os.path.join(INPUT_PATH, "val_pids_ae.npy")

SAVE_MODEL_PATH = "/content/drive/MyDrive/Diplomamunka/best_sasrec_bpr_ae_notempo.weights.h5"

# --- HIPERPARAMÉTEREK ---
BATCH_SIZE = 128
EPOCHS = 30
MAX_LEN = 50
D_MODEL = 256          # Az AE 256-dimenziós vektort ad ki → ez a természetes dimenzió!
LEARNING_RATE = 5e-5

# ==========================================
# 2. ADATOK BETÖLTÉSE
# ==========================================
print("📂 Adatok betöltése...")
ae_weights = np.load(AE_MATRIX_PATH).astype('float32')

# Padding sor hozzáadása (0-s ID = csupa nulla vektor)
padding_row = np.zeros((1, D_MODEL), dtype='float32')
embedding_matrix = np.vstack([padding_row, ae_weights])  # shape: (vocab_size+1, 256)

vocab_size = embedding_matrix.shape[0]
print(f"✅ Embedding mátrix alakja: {embedding_matrix.shape} (vocab: {vocab_size})")

train_data = np.load(TRAIN_DATA_PATH, allow_pickle=True)
val_data = np.load(VAL_DATA_PATH, allow_pickle=True)
gc.collect()

# ==========================================
# 3. HARD NEGATIVE MEDENCE (POOL) GENERÁLÁSA
# ==========================================
print("📊 Dalok népszerűségének kiszámítása...")
item_counts = np.zeros(vocab_size)
for pl in train_data:
    for item in pl:
        item_counts[item] += 1

# Word2Vec simítás (0.75 hatvány) a túl domináns slágerek ellen
smoothed_counts = np.power(item_counts, 0.75)
smoothed_counts[0] = 0.0  # A 0-s (padding) ID-t SOHA nem sorsoljuk!
p_neg = smoothed_counts / np.sum(smoothed_counts)

print("🎲 Hard Negative Pool generálása...")
NEG_POOL_SIZE = 2_000_000
hard_neg_pool = np.random.choice(np.arange(vocab_size), size=NEG_POOL_SIZE, p=p_neg)
print("✅ Pool elkészült!")

# ==========================================
# 4. GENERÁTOR
# ==========================================
class SASRecHardBPRGenerator(tf.keras.utils.Sequence):
    def __init__(self, playlists, vocab_size, neg_pool, batch_size=32, max_len=50, shuffle=True, **kwargs):
        super().__init__(**kwargs)
        self.playlists = playlists
        self.vocab_size = vocab_size
        self.neg_pool = neg_pool
        self.pool_size = len(neg_pool)
        self.batch_size = batch_size
        self.max_len = max_len
        self.shuffle = shuffle
        self.indices = np.arange(len(self.playlists))
        if self.shuffle: np.random.shuffle(self.indices)

    def __len__(self):
        return int(np.ceil(len(self.playlists) / self.batch_size))

    def __getitem__(self, index):
        batch_idx = self.indices[index * self.batch_size : (index + 1) * self.batch_size]
        batch_seq, batch_pos, batch_neg, batch_mask = [], [], [], []

        for i in batch_idx:
            pl = self.playlists[i]
            if len(pl) < 2: continue

            seq = pl[:self.max_len+1]
            input_seq = seq[:-1]
            target_pos = seq[1:]
            pad_len = self.max_len - len(input_seq)

            padded_input = [0] * pad_len + input_seq
            padded_pos = [0] * pad_len + target_pos

            pl_set = set(pl)
            padded_neg = [0] * pad_len

            for _ in range(len(target_pos)):
                neg_id = self.neg_pool[np.random.randint(0, self.pool_size)]
                while neg_id in pl_set:
                    neg_id = self.neg_pool[np.random.randint(0, self.pool_size)]
                padded_neg.append(neg_id)

            mask = [0.0] * pad_len + [1.0] * len(target_pos)

            batch_seq.append(padded_input)
            batch_pos.append(padded_pos)
            batch_neg.append(padded_neg)
            batch_mask.append(mask)

        X = {
            "seq_inputs": np.array(batch_seq),
            "pos_inputs": np.array(batch_pos),
            "neg_inputs": np.array(batch_neg)
        }
        y = np.array(batch_mask)
        return X, y

# ==========================================
# 5. BPR LOSS
# ==========================================
def bpr_loss(y_true, y_pred):
    mask = tf.cast(y_true, tf.float32)
    pos_scores = y_pred[..., 0]
    neg_scores = y_pred[..., 1]
    diff = pos_scores - neg_scores
    loss = tf.math.softplus(-diff)
    return tf.reduce_sum(loss * mask) / (tf.reduce_sum(mask) + 1e-8)

# ==========================================
# 6. MODELL
# ==========================================
def create_ae_sasrec_model(vocab_size):
    seq_inputs = layers.Input(shape=(MAX_LEN,), name="seq_inputs")
    pos_inputs = layers.Input(shape=(MAX_LEN,), name="pos_inputs")
    neg_inputs = layers.Input(shape=(MAX_LEN,), name="neg_inputs")

    item_embedding = layers.Embedding(
        vocab_size, D_MODEL, mask_zero=True,
        embeddings_initializer=tf.keras.initializers.Constant(embedding_matrix),
        trainable=True,  # Finomhangolódhat tanítás közben
        name="ae_embedding"
    )
    pos_embedding = layers.Embedding(MAX_LEN, D_MODEL, name="pos_embedding")

    x = item_embedding(seq_inputs)
    x += pos_embedding(tf.range(MAX_LEN)[tf.newaxis, :])

    # 3 blokk, 8 fej (D_MODEL=256 → key_dim=32)
    for i in range(3):
        attn = layers.MultiHeadAttention(num_heads=8, key_dim=D_MODEL//8, name=f"attn_{i}")(x, x, use_causal_mask=True)
        x = layers.LayerNormalization()(x + layers.Dropout(0.2)(attn))
        ffn = layers.Dense(D_MODEL * 4, activation='relu')(x)
        ffn = layers.Dense(D_MODEL)(ffn)
        x = layers.LayerNormalization()(x + layers.Dropout(0.2)(ffn))

    seq_states = layers.Dense(D_MODEL, name="final_projection")(x)

    pos_emb = item_embedding(pos_inputs)
    neg_emb = item_embedding(neg_inputs)

    def dot_and_cast(tensors):
        seq, emb = tensors
        scores = tf.reduce_sum(seq * emb, axis=-1, keepdims=True)
        return tf.cast(scores, tf.float32)

    dot_layer = layers.Lambda(dot_and_cast, name="dot_scoring")

    pos_scores = dot_layer([seq_states, pos_emb])
    neg_scores = dot_layer([seq_states, neg_emb])

    outputs = layers.Concatenate(axis=-1, name="concat_scores")([pos_scores, neg_scores])
    return tf.keras.Model(inputs=[seq_inputs, pos_inputs, neg_inputs], outputs=outputs)

# ==========================================
# 7. ÖSSZESZERELÉS ÉS TANÍTÁS
# ==========================================
print("🏗️ AE-alapú SASRec modell felépítése...")
model = create_ae_sasrec_model(vocab_size)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE, clipnorm=1.0), loss=bpr_loss)
model.summary()

train_gen = SASRecHardBPRGenerator(train_data, vocab_size, hard_neg_pool, batch_size=BATCH_SIZE)
val_gen   = SASRecHardBPRGenerator(val_data[::20], vocab_size, hard_neg_pool, batch_size=BATCH_SIZE)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(SAVE_MODEL_PATH, save_best_only=True, save_weights_only=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)
]

print(f"🚀 AE SASRec tanítás indítása (3 spektrogram | {EPOCHS} Epoch | Batch: {BATCH_SIZE})...")
model.fit(train_gen, validation_data=val_gen, epochs=EPOCHS, callbacks=callbacks)

gc.collect()

📂 Adatok betöltése...
✅ Embedding mátrix alakja: (27053, 256) (vocab: 27053)
📊 Dalok népszerűségének kiszámítása...
🎲 Hard Negative Pool generálása...
✅ Pool elkészült!
🏗️ AE-alapú SASRec modell felépítése...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'dot_scoring' (of type Lambda) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ seq_inputs          │ (None, 50)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ae_embedding        │ (None, 50, 256)   │  6,925,568 │ seq_inputs[0][0], │
│ (Embedding)         │                   │            │ pos_inputs[0][0], │
│                     │                   │            │ neg_inputs[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 50, 256)   │          0 │ ae_embedding[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attn_0              │ (None, 50, 256)   │    263,168 │ add[0][0],        │
│ (MultiHeadAttentio… │                   │            │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 50, 256)   │          0 │ attn_0[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 50, 256)   │          0 │ add[0][0],        │
│                     │                   │            │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 50, 256)   │        512 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 50, 1024)  │    263,168 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 50, 256)   │    262,400 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 50, 256)   │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 50, 256)   │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 50, 256)   │        512 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attn_1              │ (None, 50, 256)   │    263,168 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 50, 256)   │          0 │ attn_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 50, 256)   │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 50, 256)   │        512 │ add_3[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 50, 1024)  │    263,168 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 50, 256)   │    262,400 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 50, 256)   │          0 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 9,360,640 (35.71 MB)

 Trainable params: 9,360,640 (35.71 MB)

 Non-trainable params: 0 (0.00 B)

🚀 AE SASRec tanítás indítása (3 spektrogram | 30 Epoch | Batch: 128)...
Epoch 1/30
5520/5520 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - loss: 0.3408
Epoch 1: val_loss improved from None to 0.16326, saving model to /content/drive/MyDrive/Diplomamunka/best_sasrec_bpr_ae_notempo.weights.h5

Epoch 1: finished saving model to /content/drive/MyDrive/Diplomamunka/best_sasrec_bpr_ae_notempo.weights.h5
5520/5520 ━━━━━━━━━━━━━━━━━━━━ 342s 57ms/step - loss: 0.2430 - val_loss: 0.1633 - learning_rate: 5.0000e-05
Epoch 2/30
5520/5520 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - loss: 0.1565
Epoch 2: val_loss improved from 0.16326 to 0.14215, saving model to /content/drive/MyDrive/Diplomamunka/best_sasrec_bpr_ae_notempo.weights.h5

Epoch 2: finished saving model to /content/drive/MyDrive/Diplomamunka/best_sasrec_bpr_ae_notempo.weights.h5
5520/5520 ━━━━━━━━━━━━━━━━━━━━ 320s 58ms/step - loss: 0.1508 - val_loss: 0.1422 - learning_rate: 5.0000e-05
Epoch 3/30
5520/5520 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - loss: 0.1371
Epo

1150